<a href="https://colab.research.google.com/github/cstecker/politicsRLab/blob/main/Demokratie%20-%20Konzept%2C%20Entstehung%20und%20Stabilit%C3%A4t.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demokratie: Konzept, Entstehung und Stabilität

In diesem Notebook arbeiten wir mit Daten aus dem V-Dem-Projekt. Das Ziel ist nicht nur, Code auszuführen, sondern die einzelnen Schritte gut nachzuvollziehen.

## Lernziele

Am Ende dieses Notebooks sollten Sie:

- verstehen, wie V-Dem Demokratie misst,
- die Entstehung und Entwicklung von Demokratien über die Zeit beschreiben können,
- Unterschiede zwischen Ländern sichtbar machen können,
- und den Zusammenhang von Wohlstand und Demokratie empirisch genauer betrachten können.

## 1. Setup und Daten

Wir nutzen hier **eine kleine vorbereitete Kursdatei** statt bei jeder Sitzung das gesamte V-Dem-Paket neu zu installieren.

Das hat drei Vorteile:

- Das Notebook läuft **lokal** und in **Google Colab** mit möglichst wenig zusätzlichem Aufwand.
- Wir arbeiten nur mit den Variablen, die wir in dieser Sitzung wirklich brauchen.
- Der Code bleibt für Einsteigerinnen und Einsteiger übersichtlicher.

**Wichtig:** Für R gibt es mit `vdemdata` ein offizielles Paket von V-Dem. Eine einfache öffentliche API, die wir hier direkt im Notebook nutzen, verwenden wir dagegen nicht. Für dieses Lehr-Notebook ist die vorbereitete Kursdatei robuster.

In [ ]:
library(tidyverse)

load_vdem_data <- function() {
  local_path <- "data/vdem_democracy_teaching.rds"
  remote_url <- "https://raw.githubusercontent.com/cstecker/politicsRLab/main/data/vdem_democracy_teaching.rds"

  if (file.exists(local_path)) {
    readRDS(local_path)
  } else {
    temp_file <- tempfile(fileext = ".rds")
    download.file(remote_url, temp_file, mode = "wb")
    readRDS(temp_file)
  }
}

vdem <- load_vdem_data()

Schauen wir uns zunächst an, wie die Daten grundsätzlich aufgebaut sind. Jede Zeile steht für ein **Land in einem bestimmten Jahr**.

In [ ]:
glimpse(vdem)

Für einen ersten Blick reicht eine kleine Auswahl von Spalten. So sieht man sofort, dass der Datensatz Länder über viele Jahre hinweg beschreibt.

In [ ]:
vdem |>
  select(country_name, country_code, year, regime_type, polyarchy, liberal_democracy) |>
  slice_head(n = 10)

## 2. Was misst V-Dem?

V-Dem stellt viele verschiedene Demokratieindikatoren bereit. Für dieses Notebook konzentrieren wir uns auf einige zentrale Größen:

- `regime_type`: grobe Einordnung des politischen Regimes,
- `polyarchy`: elektorale Demokratie im Sinne freier und wettbewerblicher Wahlen,
- `liberal_democracy`: stärkere Berücksichtigung von Rechtsstaatlichkeit und Gewaltenteilung,
- `participatory_democracy`: stärkere Berücksichtigung politischer Beteiligung.

An Deutschland können wir gut sehen, dass sich diese Größen über die Zeit verändern.

In [ ]:
germany_overview <- vdem |>
  filter(country_name == "Germany", year >= 1918) |>
  select(year, regime_type, polyarchy, liberal_democracy, participatory_democracy)

bind_rows(
  slice_head(germany_overview, n = 8),
  slice_tail(germany_overview, n = 8)
)

## 3. Demokratie weltweit seit 1900

Nun betrachten wir die große Entwicklungslinie: Wie viele Länder sind im jeweiligen Jahr Autokratien oder Demokratien?

Der folgende Plot zählt für jedes Jahr, wie oft die verschiedenen Regimetypen im Datensatz vorkommen.

In [ ]:
vdem |>
  filter(year >= 1900) |>
  drop_na(regime_type) |>
  count(year, regime_type) |>
  ggplot(aes(x = year, y = n, color = regime_type)) +
  geom_line(linewidth = 1) +
  geom_vline(xintercept = c(1918, 1945, 1989), linetype = "dashed", color = "grey50") +
  labs(
    title = "Regimetypen im Zeitverlauf",
    subtitle = "Die gestrichelten Linien markieren 1918, 1945 und 1989.",
    x = "Jahr",
    y = "Anzahl der Länder",
    color = "Regimetyp"
  ) +
  theme_minimal()

Dieser Plot zeigt sehr gut, dass Demokratisierung **kein linearer Prozess** ist. Es gibt Schübe, aber auch Rückfälle.

Besonders spannend ist deshalb der Blick auf einzelne Länder.

## 4. Entstehung, Konsolidierung und Rückschritte in einzelnen Ländern

Im nächsten Schritt vergleichen wir vier Länder mit sehr unterschiedlichen Entwicklungspfaden:

- **Deutschland**: Demokratisierung, Zusammenbruch, Neuanfang,
- **Spanien**: Übergang von der Diktatur zur Demokratie,
- **Südkorea**: Demokratisierung im späten 20. Jahrhundert,
- **Ungarn**: Demokratisierung und späterer Rückschritt.

In [ ]:
selected_countries <- c("Germany", "Spain", "South Korea", "Hungary")

vdem |>
  filter(country_name %in% selected_countries, year >= 1900) |>
  ggplot(aes(x = year, y = polyarchy, color = country_name)) +
  geom_line(linewidth = 1) +
  labs(
    title = "Demokratieentwicklung ausgewählter Länder",
    x = "Jahr",
    y = "Elektoraler Demokratieindex",
    color = "Land"
  ) +
  theme_minimal()

Die Stärke solcher Zeitreihen ist, dass man **Entstehung** und **Stabilität** auseinanderhalten kann.

Ein Land kann demokratisch werden, ohne dass die Demokratie dauerhaft stabil bleibt. Genau deshalb ist es sinnvoll, auch nach Bedingungen zu fragen, die Demokratie stützen oder gefährden könnten.

### Kurze Übung

Wählen Sie nun selbst drei Länder aus und lassen Sie sich deren Entwicklung anzeigen. Ändern Sie dazu einfach die Namen im Vektor `my_countries`.

In [ ]:
my_countries <- c("Germany", "Tunisia", "Brazil")

vdem |>
  filter(country_name %in% my_countries, year >= 1900) |>
  ggplot(aes(x = year, y = polyarchy, color = country_name)) +
  geom_line(linewidth = 1) +
  labs(
    title = "Eigener Ländervergleich",
    x = "Jahr",
    y = "Elektoraler Demokratieindex",
    color = "Land"
  ) +
  theme_minimal()

## 5. Demokratie ist mehrdimensional

Demokratie ist kein einzelner Schalter, der einfach auf 0 oder 1 steht. V-Dem macht sichtbar, dass man unterschiedliche **Demokratiekonzepte** unterscheiden kann.

Im nächsten Schritt vergleichen wir für einige Länder drei Indizes im **jüngsten verfügbaren Jahr**:

- elektorale Demokratie,
- liberale Demokratie,
- partizipative Demokratie.

In [ ]:
latest_year <- max(vdem$year, na.rm = TRUE)
comparison_countries <- c(
  "Germany",
  "United States of America",
  "Hungary",
  "India",
  "China"
)

vdem |>
  filter(year == latest_year, country_name %in% comparison_countries) |>
  select(country_name, polyarchy, liberal_democracy, participatory_democracy) |>
  pivot_longer(
    cols = c(polyarchy, liberal_democracy, participatory_democracy),
    names_to = "concept",
    values_to = "index_value"
  ) |>
  mutate(
    concept = recode(
      concept,
      polyarchy = "Elektorale Demokratie",
      liberal_democracy = "Liberale Demokratie",
      participatory_democracy = "Partizipative Demokratie"
    )
  ) |>
  ggplot(aes(x = index_value, y = country_name, color = concept)) +
  geom_point(size = 3) +
  facet_wrap(~ concept) +
  labs(
    title = paste("Demokratiekonzepte im Jahr", latest_year),
    x = "Indexwert",
    y = "Land",
    color = "Konzept"
  ) +
  theme_minimal()

## 6. Ein genauerer Blick auf Wohlstand und Demokratie

Eine klassische Frage der vergleichenden Politikwissenschaft lautet: **Sind wohlhabendere Länder eher demokratisch?**

Für diese Frage nutzen wir hier die V-Dem-Variable `gdp_per_capita`. Für den in V-Dem verfügbaren Wohlstandsindikator liegen allerdings nicht für alle Jahre Werte vor. In unserer Kursdatei ist der letzte breit verfügbare Jahrgang **2019**.

Wichtig ist dabei: Ein statistischer Zusammenhang ist **noch kein Beweis für Kausalität**. Der Plot hilft uns aber, ein bekanntes theoretisches Argument empirisch zu prüfen.

In [ ]:
wealth_democracy <- vdem |>
  filter(year == 2019) |>
  drop_na(gdp_per_capita, polyarchy)

wealth_democracy |>
  ggplot(aes(x = gdp_per_capita, y = polyarchy)) +
  geom_point(alpha = 0.6, color = "steelblue") +
  geom_smooth(method = "lm", se = FALSE, color = "firebrick") +
  scale_x_log10() +
  labs(
    title = "Wohlstand und Demokratie im Jahr 2019",
    subtitle = "Die x-Achse ist logarithmisch skaliert.",
    x = "BIP pro Kopf",
    y = "Elektoraler Demokratieindex"
  ) +
  theme_minimal()

Der Zusammenhang ist insgesamt **positiv**: Wohlhabendere Länder sind im Durchschnitt häufiger demokratisch. Aber der Zusammenhang ist **nicht perfekt**.

Gerade die interessanten Fälle sind diejenigen, die **nicht genau auf der Trendlinie** liegen.

In [ ]:
interesting_countries <- c(
  "Germany",
  "United States of America",
  "India",
  "China",
  "Singapore",
  "Norway",
  "Qatar"
)

wealth_democracy |>
  ggplot(aes(x = gdp_per_capita, y = polyarchy)) +
  geom_point(alpha = 0.25, color = "grey60") +
  geom_smooth(method = "lm", se = FALSE, color = "firebrick") +
  geom_point(
    data = wealth_democracy |> filter(country_name %in% interesting_countries),
    color = "steelblue",
    size = 2.8
  ) +
  geom_text(
    data = wealth_democracy |> filter(country_name %in% interesting_countries),
    aes(label = country_name),
    nudge_y = 0.03,
    check_overlap = TRUE,
    size = 3
  ) +
  scale_x_log10() +
  labs(
    title = "Ausgewählte Länder im Wohlstand-Demokratie-Zusammenhang",
    x = "BIP pro Kopf",
    y = "Elektoraler Demokratieindex"
  ) +
  theme_minimal()

### Interpretationshilfe

Wenn Sie diesen Plot lesen, können Sie sich drei Fragen stellen:

1. Liegen wohlhabendere Länder im Durchschnitt weiter oben?
2. Welche Länder weichen deutlich vom allgemeinen Muster ab?
3. Welche politischen oder historischen Gründe könnten solche Abweichungen erklären?

Genau an dieser Stelle beginnt die eigentliche politikwissenschaftliche Interpretation.

## 7. Abschluss

Dieses Notebook hat drei Dinge zusammengeführt:

- das **Konzept** von Demokratie,
- die **Entstehung und Entwicklung** von Demokratien über die Zeit,
- und einen ersten empirischen Blick auf mögliche **Stabilisierungsbedingungen**.

Ein sinnvoller nächster Schritt wäre nun, den Zusammenhang von Wohlstand und Demokratie noch systematischer zu prüfen, zum Beispiel getrennt nach Regionen oder mit zusätzlichen Erklärungsfaktoren.